# Quickstart — generación de escenarios de potencia

Este notebook ejecuta el pipeline **power-first**:

```text
viento horario [m/s]
        ↓
curva de potencia Alto Baguales
        ↓
potencia horaria [MW]
        ↓
potencia media diaria [MW]
        ↓
TimesFM 2.5 zero-shot
        ↓
Q10, ..., Q90 de potencia
        ↓
Schaake Shuffle con histórico de potencia
        ↓
escenarios de potencia [MW]
```

El notebook no reimplementa TimesFM ni Schaake Shuffle. Utiliza los módulos del repositorio y lee el **Power Ground Truth** sólo después de completar la generación.

## 1. Entorno

Desde la raíz del repositorio:

```powershell
uv sync
```

En VS Code selecciona como kernel:

```text
.venv\Scripts\python.exe
```

Si prefieres Jupyter Lab:

```powershell
uv run --with jupyter jupyter lab
```

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image as IPythonImage, display

from wind_scenarios.data import (
    DEFAULT_DATA_PATH,
    DEFAULT_TARGET_COLUMN,
    PROJECT_ROOT,
)
from wind_scenarios.evaluation import (
    load_ground_truth,
    save_ground_truth,
)

print("Python del kernel:")
print(sys.executable)

print("\nRaíz del proyecto:")
print(PROJECT_ROOT)

print("\nDataset diario de potencia:")
print(DEFAULT_DATA_PATH)

print("\nVariable objetivo:")
print(DEFAULT_TARGET_COLUMN)

## 2. Preprocesar viento horario a potencia diaria

Esta etapa debe ejecutarse antes de TimesFM.

El archivo esperado por defecto es:

```text
data/wind_hourly.csv
```

con columnas:

```text
source_timestamp
wind_ab
```

Primero se transforma cada velocidad horaria:

\[
P_{d,h}=f(v_{d,h})
\]

y sólo después se agregan días completos de 24 horas:

\[
POWER\_MW_d=\frac{1}{24}\sum_{h=1}^{24}P_{d,h}
\]

\[
ENERGY\_MWH_d=\sum_{h=1}^{24}P_{d,h}.
\]

`POWER_MW` es la variable utilizada por TimesFM.

In [ ]:
def preprocess_power_data(
    *,
    input_path: str | Path = PROJECT_ROOT / "data" / "wind_hourly.csv",
    hourly_output: str | Path = PROJECT_ROOT / "data" / "power_ab_hourly.csv",
    daily_output: str | Path = DEFAULT_DATA_PATH,
    timestamp_column: str = "source_timestamp",
    wind_column: str = "wind_ab",
    n_turbines: float = 1.0,
    availability: float = 1.0,
    electrical_efficiency: float = 1.0,
    incomplete_day_policy: str = "error",
) -> pd.DataFrame:
    """Build the daily power dataset from hourly wind observations."""
    if incomplete_day_policy not in {"error", "drop"}:
        raise ValueError(
            "incomplete_day_policy must be 'error' or 'drop'."
        )

    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "preprocess_power_ab.py"),
        "--input",
        str(Path(input_path)),
        "--hourly_output",
        str(Path(hourly_output)),
        "--daily_output",
        str(Path(daily_output)),
        "--timestamp_column",
        timestamp_column,
        "--wind_column",
        wind_column,
        "--n_turbines",
        str(n_turbines),
        "--availability",
        str(availability),
        "--electrical_efficiency",
        str(electrical_efficiency),
        "--incomplete_day_policy",
        incomplete_day_policy,
    ]

    print("=== PREPROCESAMIENTO VIENTO -> POTENCIA ===")
    print(
        " ".join(
            f'"{part}"' if " " in part else part
            for part in command
        )
    )

    subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        check=True,
    )

    daily_output = Path(daily_output)
    daily = pd.read_csv(
        daily_output,
        parse_dates=["DATE"],
    )

    print("\nDataset diario:")
    print(daily_output)
    print("Shape:", daily.shape)

    return daily

In [ ]:
# Ejecuta esta celda sólo cuando necesites reconstruir el dataset de potencia.
#
# daily_power = preprocess_power_data(
#     n_turbines=1,
#     incomplete_day_policy="error",
# )
#
# display(daily_power.head())

## 3. Generar escenarios de potencia

La función siguiente llama a:

```text
wind_scenarios.cli.generate
```

y, cuando la generación termina correctamente, exporta `ground_truth.csv`.

Por defecto, el límite superior de potencia se lee de:

```text
data/power_ab_daily.metadata.json
```

In [ ]:
def generate_power_scenarios(
    origin: str,
    historical_method: str = "same_calendar_date",
    context_len: int = 32,
    horizon_len: int = 28,
    *,
    n_scenarios: int | None = None,
    window_radius_days: int = 15,
    min_start_distance_days: int = 0,
    selection: str = "closest",
    device: str = "auto",
    probability_grid_mode: str = "timesfm_support",
    crossing_policy: str = "error",
    power_bounds_policy: str = "clip",
    power_min_mw: float = 0.0,
    power_max_mw: float | None = None,
    export_ground_truth: bool = True,
    data_path: str | Path = DEFAULT_DATA_PATH,
    target_column: str = DEFAULT_TARGET_COLUMN,
    output_root: str | Path = PROJECT_ROOT / "outputs" / "schaake",
) -> dict:
    """Run zero-shot TimesFM + Schaake Shuffle in the power domain."""
    if historical_method not in {
        "same_calendar_date",
        "moving_window",
    }:
        raise ValueError(
            "historical_method must be "
            "'same_calendar_date' or 'moving_window'."
        )

    if context_len <= 0:
        raise ValueError("context_len must be positive.")

    if horizon_len <= 0:
        raise ValueError("horizon_len must be positive.")

    if device not in {"auto", "cpu", "cuda"}:
        raise ValueError(
            "device must be 'auto', 'cpu' or 'cuda'."
        )

    if probability_grid_mode not in {
        "timesfm_support",
        "schaake",
    }:
        raise ValueError(
            "probability_grid_mode must be "
            "'timesfm_support' or 'schaake'."
        )

    if crossing_policy not in {"error", "sort"}:
        raise ValueError(
            "crossing_policy must be 'error' or 'sort'."
        )

    if power_bounds_policy not in {
        "clip",
        "error",
        "none",
    }:
        raise ValueError(
            "power_bounds_policy must be "
            "'clip', 'error' or 'none'."
        )

    origin_ts = pd.Timestamp(origin).normalize()
    data_path = Path(data_path)
    output_root = Path(output_root)

    if not data_path.exists():
        raise FileNotFoundError(
            f"No existe el dataset diario de potencia: {data_path}. "
            "Ejecuta preprocess_power_data() primero."
        )

    command = [
        sys.executable,
        "-m",
        "wind_scenarios.cli.generate",
        "--origin",
        origin_ts.date().isoformat(),
        "--historical_method",
        historical_method,
        "--context_len",
        str(context_len),
        "--horizon_len",
        str(horizon_len),
        "--device",
        device,
        "--data_path",
        str(data_path),
        "--target_column",
        target_column,
        "--probability_grid_mode",
        probability_grid_mode,
        "--crossing_policy",
        crossing_policy,
        "--power_bounds_policy",
        power_bounds_policy,
        "--power_min_mw",
        str(power_min_mw),
        "--output_dir",
        str(output_root),
    ]

    if power_max_mw is not None:
        command.extend(
            [
                "--power_max_mw",
                str(power_max_mw),
            ]
        )

    if n_scenarios is not None:
        command.extend(
            [
                "--n_scenarios",
                str(n_scenarios),
            ]
        )

    if historical_method == "moving_window":
        command.extend(
            [
                "--window_radius_days",
                str(window_radius_days),
                "--min_start_distance_days",
                str(min_start_distance_days),
                "--selection",
                selection,
            ]
        )

    print("=== TIMESFM + SCHAAKE SHUFFLE: POTENCIA ===")
    print(
        " ".join(
            f'"{part}"' if " " in part else part
            for part in command
        )
    )
    print()

    subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        check=True,
    )

    output_dir = output_root / origin_ts.date().isoformat()

    ground_truth_path = None
    if export_ground_truth:
        ground_truth_path = save_ground_truth(
            origin=origin_ts,
            horizon_len=horizon_len,
            output_dir=output_dir,
            data_path=data_path,
            value_column=target_column,
        )

    paths = {
        "output_dir": output_dir,
        "context": output_dir / "context.csv",
        "quantiles": output_dir / "timesfm_quantiles.csv",
        "historical": output_dir / "historical_trajectories.csv",
        "metadata": output_dir / "historical_metadata.csv",
        "ranks": output_dir / "historical_ranks.csv",
        "pseudo_observations": (
            output_dir / "empirical_pseudo_observations.csv"
        ),
        "assigned_probabilities": (
            output_dir / "assigned_probabilities.csv"
        ),
        "marginals": (
            output_dir / "timesfm_discretized_marginals.csv"
        ),
        "power_scenarios": (
            output_dir / "power_scenarios.csv"
        ),
        "run_config": (
            output_dir / "run_config.json"
        ),
        "ground_truth": ground_truth_path,
    }

    result = {
        "origin": origin_ts,
        "historical_method": historical_method,
        "context_len": context_len,
        "horizon_len": horizon_len,
        "target_column": target_column,
        "output_dir": output_dir,
        "paths": paths,
        "context": pd.read_csv(
            paths["context"],
            parse_dates=["DATE"],
        ),
        "quantiles": pd.read_csv(
            paths["quantiles"],
            parse_dates=["DATE"],
        ),
        "power_scenarios": pd.read_csv(
            paths["power_scenarios"],
            index_col="scenario_id",
        ),
    }

    if ground_truth_path is not None:
        result["ground_truth"] = pd.read_csv(
            ground_truth_path,
            parse_dates=["DATE"],
        )

    print("\n=== GENERACIÓN COMPLETADA ===")
    print("Output:       ", output_dir)
    print("Context:      ", result["context"].shape)
    print("Quantiles:    ", result["quantiles"].shape)
    print("Scenarios:    ", result["power_scenarios"].shape)

    if ground_truth_path is not None:
        print("Ground truth: ", result["ground_truth"].shape)

    return result

## 4. Caso base

32 días de contexto, 28 días de horizonte y una trayectoria histórica por año.

In [ ]:
result = generate_power_scenarios(
    origin="2014-04-01",
    historical_method="same_calendar_date",
    context_len=32,
    horizon_len=28,
)

## 5. Revisar resultados

In [ ]:
print("Carpeta:")
print(result["output_dir"])

print("\nArchivos:")
for name, path in result["paths"].items():
    if path is not None:
        print(f"{name:24s} -> {path}")

print("\nContexto de potencia:")
display(result["context"].head())

print("\nCuantiles TimesFM:")
display(result["quantiles"].head())

print("\nEscenarios de potencia:")
display(result["power_scenarios"].head())

print("\nPower Ground Truth:")
display(result["ground_truth"].head())

## 6. Visualización

Los gráficos se construyen a partir de los CSV ya existentes; no vuelven a ejecutar TimesFM ni Schaake Shuffle.

Modos:

```text
single -> un escenario de potencia
fan    -> todos los escenarios
```

In [ ]:
def visualize_result(
    result: dict,
    *,
    mode: str = "single",
    scenario_number: int = 1,
    all_quantiles: bool = False,
    export_all: bool = False,
    display_plot: bool = True,
    data_path: str | Path = DEFAULT_DATA_PATH,
    target_column: str = DEFAULT_TARGET_COLUMN,
) -> Path:
    """Generate power-scenario plots from existing CSV outputs."""
    if mode not in {"single", "fan"}:
        raise ValueError(
            "mode must be 'single' or 'fan'."
        )

    if scenario_number <= 0:
        raise ValueError(
            "scenario_number must be positive."
        )

    if mode == "fan" and export_all:
        raise ValueError(
            "export_all is only used with mode='single'."
        )

    output_dir = Path(result["output_dir"])

    command = [
        sys.executable,
        "-m",
        "wind_scenarios.cli.visualize",
        "--input_dir",
        str(output_dir),
        "--data_path",
        str(Path(data_path)),
        "--target_column",
        target_column,
        "--mode",
        mode,
    ]

    if all_quantiles:
        command.append("--all_quantiles")

    if export_all:
        command.append("--export_all")
    elif mode == "single":
        command.extend(
            [
                "--scenario_number",
                str(scenario_number),
            ]
        )

    print("=== VISUALIZACIÓN ===")
    print(
        " ".join(
            f'"{part}"' if " " in part else part
            for part in command
        )
    )

    subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        check=True,
    )

    if export_all:
        plot_path = output_dir / (
            "power_scenario_plots_all_quantiles"
            if all_quantiles
            else "power_scenario_plots"
        )
        print(f"Plots guardados en: {plot_path}")
        return plot_path

    if mode == "fan":
        plot_path = output_dir / (
            "power_scenario_fan_all_quantiles.png"
            if all_quantiles
            else "power_scenario_fan.png"
        )
    else:
        suffix = (
            "_all_quantiles"
            if all_quantiles
            else ""
        )
        plot_path = (
            output_dir
            / f"scenario_{scenario_number:03d}{suffix}.png"
        )

    if not plot_path.exists():
        raise FileNotFoundError(
            f"No se creó el gráfico esperado: {plot_path}"
        )

    print(f"Gráfico guardado: {plot_path}")

    if display_plot:
        display(
            IPythonImage(
                filename=str(plot_path)
            )
        )

    return plot_path

In [ ]:
# Escenario 1 con Q10, Q50 y Q90.
scenario_plot = visualize_result(
    result,
    mode="single",
    scenario_number=1,
    all_quantiles=False,
)

In [ ]:
# Ejemplos opcionales.

# Escenario 1 con Q10-Q90:
#
# visualize_result(
#     result,
#     mode="single",
#     scenario_number=1,
#     all_quantiles=True,
# )

# Fan plot:
#
# visualize_result(
#     result,
#     mode="fan",
#     all_quantiles=True,
# )

# Exportar todos los escenarios con Q10, Q50 y Q90:
#
# visualize_result(
#     result,
#     mode="single",
#     all_quantiles=False,
#     export_all=True,
#     display_plot=False,
# )

# Exportar todos los escenarios con Q10-Q90:
#
# visualize_result(
#     result,
#     mode="single",
#     all_quantiles=True,
#     export_all=True,
#     display_plot=False,
# )

## 7. Diagnóstico de la conversión viento → potencia

La conversión física ocurre antes de TimesFM. El diagnóstico permite revisar:

- viento horario vs potencia horaria;
- observaciones transformadas sobre la curva de potencia.

In [ ]:
def plot_wind_power_preprocessing(
    *,
    days: int = 7,
    display_plots: bool = True,
) -> tuple[Path, Path]:
    command = [
        sys.executable,
        str(
            PROJECT_ROOT
            / "scripts"
            / "diagnostics"
            / "plot_wind_power_conversion.py"
        ),
        "--days",
        str(days),
    ]

    subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        check=True,
    )

    diagnostic_dir = (
        PROJECT_ROOT
        / "outputs"
        / "diagnostics"
    )

    timeseries_path = (
        diagnostic_dir
        / "hourly_wind_vs_power.png"
    )
    curve_path = (
        diagnostic_dir
        / "hourly_points_on_power_curve.png"
    )

    if display_plots:
        display(
            IPythonImage(
                filename=str(timeseries_path)
            )
        )
        display(
            IPythonImage(
                filename=str(curve_path)
            )
        )

    return timeseries_path, curve_path


# Descomenta para generar los diagnósticos.
#
# plot_wind_power_preprocessing(days=7)

## 8. Otras configuraciones

### 60 días de contexto y 60 días de forecast

In [ ]:
# result_60 = generate_power_scenarios(
#     origin="2014-04-01",
#     historical_method="same_calendar_date",
#     context_len=60,
#     horizon_len=60,
# )

### 100 escenarios con ventanas móviles

In [ ]:
# result_100 = generate_power_scenarios(
#     origin="2014-04-01",
#     historical_method="moving_window",
#     context_len=32,
#     horizon_len=28,
#     n_scenarios=100,
#     window_radius_days=15,
#     min_start_distance_days=7,
# )

## 9. Uso independiente de `evaluation.py`

Para leer el ground truth sin regenerar escenarios:

```python
ground_truth = load_ground_truth(
    origin="2014-04-01",
    horizon_len=28,
    value_column="POWER_MW",
)
```

Para guardarlo:

```python
save_ground_truth(
    origin="2014-04-01",
    horizon_len=28,
    output_dir="outputs/schaake/2014-04-01",
    value_column="POWER_MW",
)
```

El archivo contiene:

```text
DATE,POWER_MW,LEAD_DAY
```